# Análisis de Productividad PDE — Publicaciones WoS + Scopus
**Filtro:** Tributa PDE · WoS + Scopus · Article / Review · Año configurable  
**Clasificación:** Claustro + Núcleo (exactos)  
**Scope:** 7 programas definidos por código  
**Conteo:** una publicación por RUT, independiente de cuántos programas tenga el académico  
**v7** — agrega conteo Scopus y Total (sin duplicados por académico)

In [ ]:
# ================================================================
# CONFIGURACIÓN — editar aquí antes de ejecutar
# ================================================================
ANIO_ANALISIS     = 2025
CLASIFICACIONES   = ['CLAUSTRO', 'NUCLEO', 'NÚCLEO']
TIPOS_DOC_VALIDOS = ['ARTICLE', 'REVIEW']

CODIGOS_PROGRAMA  = [6163, 6436, 6556, 6566, 6706, 6806, 6883]

ARCHIVO_ACADEMICOS = 'LISTADO_ACADEMICOS_2025_MODIFICADO.xlsx'
ARCHIVO_PDE        = 'PDE.xlsx'
ARCHIVO_SALIDA     = f'resultado_productividad_pde_{ANIO_ANALISIS}_v7.xlsx'

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

try:
    display
except NameError:
    def display(df):
        print(df.to_string())

In [ ]:
# ================================================================
# UTILIDADES
# ================================================================
def normalizar_rut(x):
    if pd.isna(x): return None
    x = str(x).strip().upper()
    x = re.sub(r'[.]', '', x)
    x = re.sub(r'\s+', '', x)
    x = re.sub(r'[^0-9K\-]', '', x)
    return x if x else None

def normalizar_texto(x):
    if pd.isna(x): return None
    x = str(x).strip().upper()
    return re.sub(r'\s+', ' ', x) or None

def buscar_columna(df, candidatas, obligatoria=True, nombre_logico='columna'):
    for c in candidatas:
        if c in df.columns:
            return c
    if obligatoria:
        raise KeyError(
            f"Columna '{nombre_logico}' no encontrada.\n"
            f"Candidatas probadas: {candidatas}\n"
            f"Columnas disponibles: {list(df.columns)}"
        )
    return None

def contiene_tipo_valido(x, tipos_validos):
    txt = normalizar_texto(x)
    if not txt: return False
    tokens = [t.strip() for t in re.split(r'[;,|/]+', txt) if t.strip()]
    return any(t in tipos_validos for t in tokens)

def qa(titulo, valor, umbral_warn=None):
    icono = '⚠️ ' if (umbral_warn is not None and valor > umbral_warn) else '✅'
    print(f'  {icono}  {titulo}: {valor}')

In [ ]:
# ================================================================
# 1. CARGA DE DATOS
# ================================================================
for archivo in [ARCHIVO_ACADEMICOS, ARCHIVO_PDE]:
    if not Path(archivo).exists():
        raise FileNotFoundError(
            f"\n❌ Archivo no encontrado: '{archivo}'\n"
            f"   Carpeta actual: {Path('.').resolve()}"
        )

print('📂 Cargando archivos...')

data    = pd.read_excel(ARCHIVO_ACADEMICOS)
autores = pd.read_excel(ARCHIVO_PDE, sheet_name='06_Autores_Publicacion')
pubs    = pd.read_excel(ARCHIVO_PDE, sheet_name='05_Publicaciones_Unicas')
wos     = pd.read_excel(ARCHIVO_PDE, sheet_name='01_WoS_Base')
scopus  = pd.read_excel(ARCHIVO_PDE, sheet_name='02_Scopus_Base')

try:
    indicadores = pd.read_excel(ARCHIVO_PDE, sheet_name='09_Indicadores_Publicacion')
except Exception:
    indicadores = pd.DataFrame()
    print('  ⚠️  Hoja 09_Indicadores_Publicacion no encontrada.')

print(f'  Académicos (todas las filas) : {len(data):,}')
print(f'  Autores PDE                  : {len(autores):,}')
print(f'  Pubs únicas                  : {len(pubs):,}')
print(f'  Registros WoS                : {len(wos):,}')
print(f'  Registros Scopus             : {len(scopus):,}')
print()

col_rut_data    = buscar_columna(data,    ['RUT/Pasaporte', 'RUT', 'Rut'],                          nombre_logico='RUT académicos')
col_clasif      = buscar_columna(data,    ['Clasificación Académico', 'Clasificacion Academico', 'Clasificación'], nombre_logico='Clasificación')
col_nombre      = buscar_columna(data,    ['Nombre Académico', 'Nombre Consolidado', 'Nombre'],     obligatoria=False, nombre_logico='Nombre')
col_prog        = buscar_columna(data,    ['Nombre Programa', 'Programa'],                           obligatoria=False, nombre_logico='Programa')
col_cod_prog    = buscar_columna(data,    ['Código Programa', 'Codigo Programa'],                    obligatoria=False, nombre_logico='Código Programa')
col_rut_autores = buscar_columna(autores, ['Rut', 'RUT', 'rut'],                                    nombre_logico='RUT autores PDE')

print('  Columnas detectadas:')
print(f'    RUT académicos   → {col_rut_data}')
print(f'    Clasificación    → {col_clasif}')
print(f'    Nombre           → {col_nombre}')
print(f'    Programa         → {col_prog}')
print(f'    Código Programa  → {col_cod_prog}')
print(f'    RUT autores PDE  → {col_rut_autores}')

In [ ]:
# ================================================================
# 2. VALIDACIÓN DE CALIDAD DE DATOS (QA)
# ================================================================
print('\n🔍 Control de calidad\n')

data['rut_clean']    = data[col_rut_data].apply(normalizar_rut)
autores['rut_clean'] = autores[col_rut_autores].apply(normalizar_rut)

qa('RUTs nulos en académicos',      data['rut_clean'].isna().sum(),                umbral_warn=0)
qa('RUTs nulos en autores PDE',     autores['rut_clean'].isna().sum(),             umbral_warn=0)
qa('RUTs duplicados en académicos', data['rut_clean'].dropna().duplicated().sum(), umbral_warn=0)

print()
print('  Clasificaciones en el listado completo:')
for k, v in data[col_clasif].value_counts().items():
    print(f'    {k}: {v}')

# Filtro 1: Clasificación Claustro/Núcleo exactos
data['clasif_norm'] = data[col_clasif].apply(normalizar_texto)
_clasifs_upper = [c.upper() for c in CLASIFICACIONES]
data_clasif = data[data['clasif_norm'].isin(_clasifs_upper)].copy()

# Filtro 2: Solo los programas del scope
if col_cod_prog:
    data['cod_prog_num'] = pd.to_numeric(data[col_cod_prog], errors='coerce')
    academicos_filtrados = data[
        data['clasif_norm'].isin(_clasifs_upper) &
        data['cod_prog_num'].isin(CODIGOS_PROGRAMA)
    ].copy()
else:
    print('  ⚠️  No se encontró columna de Código Programa — se usa solo filtro de clasificación')
    academicos_filtrados = data_clasif.copy()

print()
qa('Registros tras filtro clasificación + programa', len(academicos_filtrados))
qa('RUTs únicos en scope',                           academicos_filtrados['rut_clean'].nunique())

In [ ]:
# ================================================================
# 3. CONSTRUCCIÓN DEL DATASET PDE
# ================================================================
for df_tmp in [autores, pubs]:
    df_tmp.rename(columns={'DOI_Normalizado': 'DOI_Pub', 'Titulo_Normalizado': 'Titulo_Pub'},
                  inplace=True, errors='ignore')

cols_pubs = [c for c in [
    'ID_Cluster_Publicacion', 'Fuente_Principal', 'ID_Fuente_Principal',
    'DOI_Pub', 'Titulo_Pub', 'Publicacion_Tributa_PDE'
] if c in pubs.columns]

pde = autores.merge(pubs[cols_pubs].drop_duplicates(), on='ID_Cluster_Publicacion', how='left')

if 'Publicacion_Tributa_PDE' not in pde.columns:
    if not indicadores.empty and 'Publicacion_Tributa_PDE' in indicadores.columns:
        pde = pde.merge(
            indicadores[['ID_Cluster_Publicacion', 'Publicacion_Tributa_PDE']].drop_duplicates(),
            on='ID_Cluster_Publicacion', how='left'
        )
    else:
        pde['Publicacion_Tributa_PDE'] = pd.NA

# --- Metadatos WoS ---
col_wos_id   = buscar_columna(wos, ['UT_WOS', 'UT'],               obligatoria=False, nombre_logico='ID WoS')
col_wos_year = buscar_columna(wos, ['Publication_Year', 'PY'],      obligatoria=False, nombre_logico='Año WoS')
col_wos_type = buscar_columna(wos, ['Document_Type', 'DT'],         obligatoria=False, nombre_logico='Tipo WoS')

# --- Metadatos Scopus ---
col_sc_id    = buscar_columna(scopus, ['EID'],                                 obligatoria=False, nombre_logico='ID Scopus')
col_sc_year  = buscar_columna(scopus, ['Year', 'Publication_Year'],             obligatoria=False, nombre_logico='Año Scopus')
col_sc_type  = buscar_columna(scopus, ['Document_Type', 'SubtypeDescription'],  obligatoria=False, nombre_logico='Tipo Scopus')

frames_meta = []
if col_wos_id:
    wm = wos[[c for c in [col_wos_id, col_wos_year, col_wos_type] if c]].copy()
    wm.rename(columns={col_wos_id: 'ID_Fuente_Principal', col_wos_year: 'anio_pub',
                        col_wos_type: 'tipo_documento_meta'}, inplace=True)
    wm['schema_meta'] = 'WOS'
    frames_meta.append(wm)

if col_sc_id:
    sm = scopus[[c for c in [col_sc_id, col_sc_year, col_sc_type] if c]].copy()
    sm.rename(columns={col_sc_id: 'ID_Fuente_Principal', col_sc_year: 'anio_pub',
                        col_sc_type: 'tipo_documento_meta'}, inplace=True)
    sm['schema_meta'] = 'SCOPUS'
    frames_meta.append(sm)

if frames_meta:
    meta = pd.concat(frames_meta, ignore_index=True)
    meta = meta.dropna(subset=['ID_Fuente_Principal'])
    meta['_prio'] = meta['schema_meta'].map({'WOS': 0, 'SCOPUS': 1}).fillna(9)
    meta = (meta.sort_values(['ID_Fuente_Principal', '_prio'])
                .drop_duplicates('ID_Fuente_Principal', keep='first')
                .drop(columns='_prio'))
    pde = pde.merge(meta, on='ID_Fuente_Principal', how='left')
else:
    pde['anio_pub'] = pd.NA
    pde['tipo_documento_meta'] = pd.NA
    pde['schema_meta'] = pd.NA

pde['anio_pub'] = pd.to_numeric(pde['anio_pub'], errors='coerce')

In [ ]:
# ================================================================
# 4. FILTRADO BASE (tributa + año + tipo documento)
# ================================================================
pde['tributa_norm'] = pde['Publicacion_Tributa_PDE'].apply(normalizar_texto)

if 'Fuente_Principal' in pde.columns:
    pde['fuente_final_norm'] = pde['Fuente_Principal'].apply(normalizar_texto)
else:
    pde['fuente_final_norm'] = pd.NA

mask_vacia = pde['fuente_final_norm'].isna()
pde.loc[mask_vacia, 'fuente_final_norm'] = (
    pde.loc[mask_vacia, 'schema_meta'].apply(normalizar_texto)
)

pde['es_tipo_valido'] = pde['tipo_documento_meta'].apply(
    lambda x: contiene_tipo_valido(x, set(TIPOS_DOC_VALIDOS))
)

# Filtro base: tributa + año + tipo
pde_base = pde[
    pde['tributa_norm'].isin(['SI', 'SÍ'])
    & (pde['anio_pub'] == ANIO_ANALISIS)
    & pde['es_tipo_valido']
].copy()

# --- Sub-filtro WoS ---
FUENTES_WOS    = ['WOS', 'WEB OF SCIENCE']
patron_wos     = '|'.join(re.escape(f) for f in FUENTES_WOS)
pde_base['es_wos']    = pde_base['fuente_final_norm'].fillna('').str.contains(patron_wos, regex=True)

# --- Sub-filtro Scopus ---
FUENTES_SCOPUS = ['SCOPUS']
patron_scopus  = '|'.join(re.escape(f) for f in FUENTES_SCOPUS)
pde_base['es_scopus'] = pde_base['fuente_final_norm'].fillna('').str.contains(patron_scopus, regex=True)

pde_wos    = pde_base[pde_base['es_wos']].copy()
pde_scopus = pde_base[pde_base['es_scopus']].copy()

print(f'📊 Registros tras filtro PDE (año {ANIO_ANALISIS} | Article/Review | Tributa = Sí):')
print(f'   WoS    — registros: {len(pde_wos):,}  |  pubs únicas: {pde_wos["ID_Cluster_Publicacion"].nunique():,}  |  autores únicos: {pde_wos["rut_clean"].nunique():,}')
print(f'   Scopus — registros: {len(pde_scopus):,}  |  pubs únicas: {pde_scopus["ID_Cluster_Publicacion"].nunique():,}  |  autores únicos: {pde_scopus["rut_clean"].nunique():,}')
print(f'   Total  — pubs únicas (unión): {len(set(pde_wos["ID_Cluster_Publicacion"]) | set(pde_scopus["ID_Cluster_Publicacion"])):,}')

In [ ]:
# ================================================================
# 5. CONTEO POR ACADÉMICO
# ================================================================
def contar_pubs_por_rut(df_filtrado, nombre_col):
    """Una publicación cuenta una sola vez por RUT."""
    return (
        df_filtrado
        .dropna(subset=['rut_clean', 'ID_Cluster_Publicacion'])
        .drop_duplicates(subset=['rut_clean', 'ID_Cluster_Publicacion'])
        .groupby('rut_clean', as_index=False)
        .size()
        .rename(columns={'size': nombre_col})
    )

n_wos    = contar_pubs_por_rut(pde_wos,    'n_pub_WoS')
n_scopus = contar_pubs_por_rut(pde_scopus, 'n_pub_Scopus')

# Total sin duplicados: unión de IDs por RUT
pde_union = pde_base[pde_base['es_wos'] | pde_base['es_scopus']].copy()
n_total   = contar_pubs_por_rut(pde_union, 'n_pub_Total')

# Unir todo al scope de académicos filtrados
resultado = academicos_filtrados.copy()
resultado = resultado.merge(n_wos,    on='rut_clean', how='left')
resultado = resultado.merge(n_scopus, on='rut_clean', how='left')
resultado = resultado.merge(n_total,  on='rut_clean', how='left')

resultado['n_pub_WoS']    = resultado['n_pub_WoS'].fillna(0).astype(int)
resultado['n_pub_Scopus'] = resultado['n_pub_Scopus'].fillna(0).astype(int)
resultado['n_pub_Total']  = resultado['n_pub_Total'].fillna(0).astype(int)
resultado['tiene_publicaciones'] = resultado['n_pub_Total'] > 0

# Resumen a nivel RUT único
resumen_rut = resultado.drop_duplicates(subset=['rut_clean'])
total       = resumen_rut['rut_clean'].nunique()
con_pub     = resumen_rut['tiene_publicaciones'].sum()
sin_pub     = total - con_pub

print(f'📋 RESUMEN — año {ANIO_ANALISIS}')
print(f'   Total académicos únicos en scope : {total}')
print(f'   Con al menos 1 pub (WoS o Scopus): {con_pub}  ({con_pub/total*100:.1f}%)')
print(f'   Sin publicaciones                : {sin_pub}  ({sin_pub/total*100:.1f}%)')
print(f'   Total pubs WoS (suma por académico)   : {resumen_rut["n_pub_WoS"].sum()}')
print(f'   Total pubs Scopus (suma por académico): {resumen_rut["n_pub_Scopus"].sum()}')
print(f'   Total pubs sin duplicados             : {resumen_rut["n_pub_Total"].sum()}')
print()

print('🏆 Top 15 por productividad total:')
cols_top = [c for c in [col_nombre, col_prog, 'clasif_norm', 'n_pub_WoS', 'n_pub_Scopus', 'n_pub_Total']
            if c and c in resumen_rut.columns]
display(
    resumen_rut[cols_top]
    .sort_values('n_pub_Total', ascending=False)
    .head(15)
    .reset_index(drop=True)
)

In [ ]:
# ================================================================
# 6. DETALLE DE AUDITORÍA
# ================================================================
detalle_cols = [c for c in [
    'rut_clean', col_nombre, 'ID_Cluster_Publicacion', 'Titulo_Pub', 'DOI_Pub',
    'Fuente_Principal', 'fuente_final_norm', 'schema_meta',
    'tipo_documento_meta', 'anio_pub', 'Publicacion_Tributa_PDE'
] if c and c in pde_union.columns]

detalle_debug = (
    pde_union[detalle_cols]
    .drop_duplicates()
    .sort_values(['rut_clean', 'anio_pub', 'ID_Cluster_Publicacion'], na_position='last')
    .copy()
)

In [ ]:
# ================================================================
# 7. EXPORTAR EXCEL FORMATEADO
# ================================================================
cols_export = [c for c in [
    col_nombre, col_prog, 'clasif_norm', 'rut_clean',
    'n_pub_WoS', 'n_pub_Scopus', 'n_pub_Total', 'tiene_publicaciones'
] if c and c in resumen_rut.columns]

df_export = resumen_rut[cols_export].sort_values('n_pub_Total', ascending=False).copy()
df_export.columns = [c.replace('_', ' ').title() for c in df_export.columns]

with pd.ExcelWriter(ARCHIVO_SALIDA, engine='openpyxl') as writer:
    df_export.to_excel(writer, sheet_name='Resultado', index=False)
    detalle_debug.to_excel(writer, sheet_name='Detalle_Auditoria', index=False)

    wb_out = writer.book
    thin   = Side(style='thin', color='BFBFBF')
    borde  = Border(left=thin, right=thin, top=thin, bottom=thin)

    fill_header  = PatternFill('solid', fgColor='1F3864')
    fill_con_pub = PatternFill('solid', fgColor='C6EFCE')
    fill_sin_pub = PatternFill('solid', fgColor='FCE4D6')
    font_header  = Font(bold=True, color='FFFFFF', size=11)
    font_bold    = Font(bold=True)
    a_center     = Alignment(horizontal='center', vertical='center', wrap_text=True)
    a_left       = Alignment(horizontal='left',   vertical='center')

    ws = wb_out['Resultado']

    # Detectar columnas de conteo
    idx_wos = idx_scopus = idx_total = idx_tiene = None
    for idx, cell in enumerate(ws[1], 1):
        v = str(cell.value or '').lower()
        if 'wos' in v and 'scopus' not in v and 'total' not in v: idx_wos    = idx
        if 'scopus' in v:                                          idx_scopus = idx
        if 'total' in v:                                           idx_total  = idx
        if 'tiene' in v:                                           idx_tiene  = idx

    for cell in ws[1]:
        cell.fill = fill_header
        cell.font = font_header
        cell.alignment = a_center
        cell.border = borde

    n_filas = len(df_export)
    for row in ws.iter_rows(min_row=2, max_row=n_filas + 1):
        tiene_pub = False
        if idx_tiene:
            val = str(row[idx_tiene - 1].value or '').upper()
            tiene_pub = val in ('TRUE', 'VERDADERO', '1', 'SÍ', 'SI')
        fill_row = fill_con_pub if tiene_pub else fill_sin_pub
        for cell in row:
            cell.fill = fill_row
            cell.border = borde
            cell.alignment = a_center if cell.column in [idx_wos, idx_scopus, idx_total] else a_left
        # Negrita en totales
        for col_idx in [idx_wos, idx_scopus, idx_total]:
            if col_idx:
                nc = row[col_idx - 1]
                if isinstance(nc.value, (int, float)) and nc.value > 0:
                    nc.font = font_bold

    for col in ws.columns:
        max_len = max((len(str(c.value or '')) for c in col), default=10)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 4, 45)

    # Pie de página con resumen
    lr = n_filas + 3
    ws.cell(lr,   1, f'Total académicos únicos en scope: {total}').font = font_bold
    ws.cell(lr+1, 1, f'Con publicaciones {ANIO_ANALISIS} (WoS o Scopus): {con_pub}').font = font_bold
    ws.cell(lr+2, 1, f'Sin publicaciones: {sin_pub}').font = font_bold
    ws.freeze_panes = 'A2'

    ws2 = wb_out['Detalle_Auditoria']
    for cell in ws2[1]:
        cell.fill = PatternFill('solid', fgColor='2E4057')
        cell.font = Font(bold=True, color='FFFFFF')
        cell.alignment = a_center
    for col in ws2.columns:
        max_len = max((len(str(c.value or '')) for c in col), default=10)
        ws2.column_dimensions[get_column_letter(col[0].column)].width = min(max_len + 3, 50)
    ws2.freeze_panes = 'A2'

print(f'✅ Archivo generado: {ARCHIVO_SALIDA}')
print(f'   "Resultado"         → {total} académicos únicos')
print(f'   "Detalle_Auditoria" → {len(detalle_debug)} registros')
print()
print('   🟢 Verde  = académico con publicaciones (WoS y/o Scopus)')
print('   🔴 Salmón = académico sin publicaciones')